In [79]:
!pip install pytorch-crf -q

In [80]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF

SEED = 42

EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

WORD_EMBEDDING_DIM = 100
CHAR_EMBEDDING_DIM = 30
CHAR_CHANNELS = 50
HIDDEN_DIM = 128
DROPOUT = 0.3

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [81]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)

Seed: 42


In [82]:
datasets = {
    "Barishal": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Barishal_NER.csv"),
    "Chittagong": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Chittagong_NER.csv"),
    "Mymensingh": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Mymensingh_NER.csv"),
    "Noakhali": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Noakhali_NER.csv"),
    "Sylhet": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Sylhet_NER.csv")
}

print("Datasets loaded.")

Datasets loaded.


In [83]:
sentence_data = {}

for dialect, df in datasets.items():

    df["Sentence #"] = df["Sentence #"].ffill()

    word_column = [
        c for c in df.columns
        if c.endswith("_word")
    ][0]

    sentences = []

    for sentence_id, group in df.groupby("Sentence #"):

        sentences.append({
            "sentence_id": int(
                str(sentence_id).replace("Sentence: ", "")
            ),
            "tokens": group[word_column].astype(str).tolist(),
            "tags": group["bio_tag"].astype(str).tolist()
        })

    sentence_data[dialect] = sentences

for dialect in sentence_data:
    print(
        dialect,
        "| Sentences:",
        len(sentence_data[dialect])
    )

Barishal | Sentences: 3481
Chittagong | Sentences: 3481
Mymensingh | Sentences: 3481
Noakhali | Sentences: 3481
Sylhet | Sentences: 3481


In [84]:
tags = set()

for sentences in sentence_data.values():

    for sentence in sentences:
        tags.update(sentence["tags"])

tags = sorted(tags)

tag_to_id = {
    tag: i
    for i, tag in enumerate(tags)
}

id_to_tag = {
    i: tag
    for tag, i in tag_to_id.items()
}

print("Number of tags:", len(tags))
print(tag_to_id)

Number of tags: 20
{'\nI-LOC': 0, 'B-ANI': 1, 'B-COL': 2, 'B-FOOD': 3, 'B-LOC': 4, 'B-OBJ': 5, 'B-ORG': 6, 'B-PER': 7, 'B-REL': 8, 'B-ROLE': 9, 'I-ANI': 10, 'I-COL': 11, 'I-FOOD': 12, 'I-LOC': 13, 'I-OBJ': 14, 'I-ORG': 15, 'I-PER': 16, 'I-REL': 17, 'I-ROLE': 18, 'O': 19}


In [85]:
def get_cross_dialect_data(held_out):

    train_sentences = []
    test_sentences = sentence_data[held_out]

    for dialect in sentence_data:

        if dialect != held_out:
            train_sentences.extend(
                sentence_data[dialect]
            )

    return train_sentences, test_sentences

In [86]:
def build_word_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sentence in sentences:

        for token in sentence["tokens"]:

            if token not in vocab:
                vocab[token] = len(vocab)

    return vocab

In [87]:
def build_char_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for sentence in sentences:

        for word in sentence["tokens"]:

            for char in word:

                if char not in vocab:
                    vocab[char] = len(vocab)

    return vocab

In [88]:
def encode_sentence(sentence, vocab):

    word_ids = [
        vocab.get(
            token,
            vocab["<UNK>"]
        )
        for token in sentence["tokens"]
    ]

    tag_ids = [
        tag_to_id[tag]
        for tag in sentence["tags"]
    ]

    return word_ids, tag_ids


def encode_characters(sentence, char_vocab):

    char_ids = []

    for word in sentence["tokens"]:

        chars = [
            char_vocab.get(
                char,
                char_vocab["<UNK>"]
            )
            for char in word
        ]

        char_ids.append(chars)

    return char_ids

In [89]:
class NERDataset(Dataset):

    def __init__(self, sentences, vocab):

        self.data = [
            encode_sentence(sentence, vocab)
            for sentence in sentences
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        words, tags = self.data[index]

        return (
            torch.tensor(words, dtype=torch.long),
            torch.tensor(tags, dtype=torch.long)
        )

In [90]:
def collate_fn(batch):

    words, tags = zip(*batch)

    max_length = max(
        len(x) for x in words
    )

    padded_words = []
    padded_tags = []
    masks = []

    for word_ids, tag_ids in zip(words, tags):

        padding = max_length - len(word_ids)

        padded_words.append(
            torch.cat([
                word_ids,
                torch.zeros(
                    padding,
                    dtype=torch.long
                )
            ])
        )

        padded_tags.append(
            torch.cat([
                tag_ids,
                torch.zeros(
                    padding,
                    dtype=torch.long
                )
            ])
        )

        masks.append(
            torch.tensor(
                [1] * len(word_ids) +
                [0] * padding,
                dtype=torch.bool
            )
        )

    return (
        torch.stack(padded_words),
        torch.stack(padded_tags),
        torch.stack(masks)
    )

In [91]:
class CharNERDataset(Dataset):

    def __init__(
        self,
        sentences,
        word_vocab,
        char_vocab
    ):

        self.data = []

        for sentence in sentences:

            word_ids, tag_ids = encode_sentence(
                sentence,
                word_vocab
            )

            char_ids = encode_characters(
                sentence,
                char_vocab
            )

            self.data.append(
                (
                    word_ids,
                    char_ids,
                    tag_ids
                )
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        return self.data[index]

In [92]:
def char_collate_fn(batch):

    words, chars, tags = zip(*batch)

    max_sentence_length = max(
        len(x) for x in words
    )

    max_word_length = max(
        len(char)
        for sentence in chars
        for char in sentence
    )

    padded_words = []
    padded_tags = []
    padded_chars = []
    masks = []

    for word_ids, char_ids, tag_ids in zip(
        words,
        chars,
        tags
    ):

        sentence_length = len(word_ids)
        sentence_padding = (
            max_sentence_length -
            sentence_length
        )

        padded_words.append(
            word_ids +
            [0] * sentence_padding
        )

        padded_tags.append(
            tag_ids +
            [0] * sentence_padding
        )

        masks.append(
            [1] * sentence_length +
            [0] * sentence_padding
        )

        sentence_chars = []

        for word in char_ids:

            word_padding = (
                max_word_length -
                len(word)
            )

            sentence_chars.append(
                word +
                [0] * word_padding
            )

        for _ in range(sentence_padding):

            sentence_chars.append(
                [0] * max_word_length
            )

        padded_chars.append(sentence_chars)

    return (
        torch.tensor(
            padded_words,
            dtype=torch.long
        ),
        torch.tensor(
            padded_chars,
            dtype=torch.long
        ),
        torch.tensor(
            padded_tags,
            dtype=torch.long
        ),
        torch.tensor(
            masks,
            dtype=torch.bool
        )
    )

In [93]:
class BiLSTMCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        tag_size
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            WORD_EMBEDDING_DIM,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=WORD_EMBEDDING_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.fc = nn.Linear(
            HIDDEN_DIM * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(
        self,
        tokens,
        tags=None,
        mask=None
    ):

        x = self.embedding(tokens)

        x, _ = self.lstm(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            return -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(
            emissions,
            mask=mask
        )

In [94]:
class BiGRUCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        tag_size
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            WORD_EMBEDDING_DIM,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=WORD_EMBEDDING_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.fc = nn.Linear(
            HIDDEN_DIM * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(
        self,
        tokens,
        tags=None,
        mask=None
    ):

        x = self.embedding(tokens)

        x, _ = self.gru(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            return -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(
            emissions,
            mask=mask
        )

In [95]:
class CharCNNBiLSTMCRF(nn.Module):

    def __init__(
        self,
        word_vocab_size,
        char_vocab_size,
        tag_size
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            word_vocab_size,
            WORD_EMBEDDING_DIM,
            padding_idx=0
        )

        self.char_embedding = nn.Embedding(
            char_vocab_size,
            CHAR_EMBEDDING_DIM,
            padding_idx=0
        )

        self.char_cnn = nn.Conv1d(
            in_channels=CHAR_EMBEDDING_DIM,
            out_channels=CHAR_CHANNELS,
            kernel_size=3,
            padding=1
        )

        self.lstm = nn.LSTM(
            input_size=
                WORD_EMBEDDING_DIM +
                CHAR_CHANNELS,
            hidden_size=HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.fc = nn.Linear(
            HIDDEN_DIM * 2,
            tag_size
        )

        self.crf = CRF(
            tag_size,
            batch_first=True
        )

    def forward(
        self,
        words,
        chars,
        tags=None,
        mask=None
    ):

        word_emb = self.word_embedding(words)

        char_emb = self.char_embedding(chars)

        batch_size = chars.size(0)
        sentence_length = chars.size(1)
        word_length = chars.size(2)

        char_emb = char_emb.view(
            batch_size * sentence_length,
            word_length,
            CHAR_EMBEDDING_DIM
        )

        char_emb = char_emb.transpose(1, 2)

        char_features = self.char_cnn(
            char_emb
        )

        char_features = torch.max(
            char_features,
            dim=2
        ).values

        char_features = char_features.view(
            batch_size,
            sentence_length,
            CHAR_CHANNELS
        )

        x = torch.cat(
            [
                word_emb,
                char_features
            ],
            dim=2
        )

        x, _ = self.lstm(x)

        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:

            return -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

        return self.crf.decode(
            emissions,
            mask=mask
        )

In [96]:
def get_entities(tags):

    entities = []

    start = None
    entity_type = None

    for i, tag in enumerate(tags + ["O"]):

        if tag.startswith("B-"):

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = i
            entity_type = tag[2:]

        elif tag.startswith("I-"):

            if start is None:
                start = i
                entity_type = tag[2:]

        else:

            if start is not None:
                entities.append(
                    (start, i - 1, entity_type)
                )

            start = None
            entity_type = None

    return set(entities)

In [97]:
def train_word_model(
    model,
    loader
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for words, tags, mask in loader:

            words = words.to(device)
            tags = tags.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            loss = model(
                words,
                tags=tags,
                mask=mask
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = (
            total_loss /
            len(loader)
        )

        print(
            f"Epoch {epoch + 1}/{EPOCHS}"
            f" - Loss: {avg_loss:.4f}"
        )

In [98]:
def train_char_model(
    model,
    loader
):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for words, chars, tags, mask in loader:

            words = words.to(device)
            chars = chars.to(device)
            tags = tags.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            loss = model(
                words,
                chars,
                tags=tags,
                mask=mask
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = (
            total_loss /
            len(loader)
        )

        print(
            f"Epoch {epoch + 1}/{EPOCHS}"
            f" - Loss: {avg_loss:.4f}"
        )

In [99]:
def evaluate_word_model(
    model,
    loader
):

    model.eval()

    correct = 0
    predicted = 0
    actual = 0

    with torch.no_grad():

        for words, tags, mask in loader:

            words = words.to(device)
            mask = mask.to(device)

            predictions = model(
                words,
                mask=mask
            )

            for i, pred in enumerate(
                predictions
            ):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                true_entities = get_entities(
                    true_tags
                )

                pred_entities = get_entities(
                    pred_tags
                )

                correct += len(
                    true_entities &
                    pred_entities
                )

                actual += len(true_entities)
                predicted += len(pred_entities)

    precision = (
        correct / predicted
        if predicted else 0
    )

    recall = (
        correct / actual
        if actual else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall else 0
    )

    return precision, recall, f1

In [100]:
def evaluate_char_model(
    model,
    loader
):

    model.eval()

    correct = 0
    predicted = 0
    actual = 0

    with torch.no_grad():

        for words, chars, tags, mask in loader:

            words = words.to(device)
            chars = chars.to(device)
            mask = mask.to(device)

            predictions = model(
                words,
                chars,
                mask=mask
            )

            for i, pred in enumerate(
                predictions
            ):

                length = mask[i].sum().item()

                true_tags = [
                    id_to_tag[x.item()]
                    for x in tags[i][:length]
                ]

                pred_tags = [
                    id_to_tag[x]
                    for x in pred
                ]

                true_entities = get_entities(
                    true_tags
                )

                pred_entities = get_entities(
                    pred_tags
                )

                correct += len(
                    true_entities &
                    pred_entities
                )

                actual += len(true_entities)
                predicted += len(pred_entities)

    precision = (
        correct / predicted
        if predicted else 0
    )

    recall = (
        correct / actual
        if actual else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall else 0
    )

    return precision, recall, f1

In [101]:
all_results = []

dialects = list(sentence_data.keys())

for held_out in dialects:

    print("\n")
    print("=" * 70)
    print("HELD-OUT DIALECT:", held_out)
    print("=" * 70)

    # Create train/test data

    train_sentences, test_sentences = (
        get_cross_dialect_data(held_out)
    )

    print(
        "Training sentences:",
        len(train_sentences)
    )

    print(
        "Test sentences:",
        len(test_sentences)
    )

    # Build training-only vocabularies

    word_vocab = build_word_vocab(
        train_sentences
    )

    char_vocab = build_char_vocab(
        train_sentences
    )

    print(
        "Word vocabulary:",
        len(word_vocab)
    )

    print(
        "Character vocabulary:",
        len(char_vocab)
    )


    # Word datasets/loaders


    train_dataset = NERDataset(
        train_sentences,
        word_vocab
    )

    test_dataset = NERDataset(
        test_sentences,
        word_vocab
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )


    # Character datasets/loaders

    char_train_dataset = CharNERDataset(
        train_sentences,
        word_vocab,
        char_vocab
    )

    char_test_dataset = CharNERDataset(
        test_sentences,
        word_vocab,
        char_vocab
    )

    char_train_loader = DataLoader(
        char_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=char_collate_fn
    )

    char_test_loader = DataLoader(
        char_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=char_collate_fn
    )

    # MODEL 1: BiLSTM-CRF

    print("\n--- BiLSTM-CRF ---")

    model = BiLSTMCRF(
        vocab_size=len(word_vocab),
        tag_size=len(tag_to_id)
    ).to(device)

    train_word_model(
        model,
        train_loader
    )

    precision, recall, f1 = (
        evaluate_word_model(
            model,
            test_loader
        )
    )

    all_results.append({
        "Held-out Dialect": held_out,
        "Model": "BiLSTM-CRF",
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    print(
        f"F1: {f1:.4f}"
    )


    # MODEL 2: BiGRU-CRF

    print("\n--- BiGRU-CRF ---")

    model = BiGRUCRF(
        vocab_size=len(word_vocab),
        tag_size=len(tag_to_id)
    ).to(device)

    train_word_model(
        model,
        train_loader
    )

    precision, recall, f1 = (
        evaluate_word_model(
            model,
            test_loader
        )
    )

    all_results.append({
        "Held-out Dialect": held_out,
        "Model": "BiGRU-CRF",
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    print(
        f"F1: {f1:.4f}"
    )


    # MODEL 3: Char-CNN-BiLSTM-CRF

    print("\n--- Char-CNN-BiLSTM-CRF ---")

    model = CharCNNBiLSTMCRF(
        word_vocab_size=len(word_vocab),
        char_vocab_size=len(char_vocab),
        tag_size=len(tag_to_id)
    ).to(device)

    train_char_model(
        model,
        char_train_loader
    )

    precision, recall, f1 = (
        evaluate_char_model(
            model,
            char_test_loader
        )
    )

    all_results.append({
        "Held-out Dialect": held_out,
        "Model": "Char-CNN-BiLSTM-CRF",
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    print(
        f"F1: {f1:.4f}"
    )



HELD-OUT DIALECT: Barishal
Training sentences: 13924
Test sentences: 3481
Word vocabulary: 12305
Character vocabulary: 88

--- BiLSTM-CRF ---
Epoch 1/20 - Loss: 3.3989
Epoch 2/20 - Loss: 1.6710
Epoch 3/20 - Loss: 1.0794
Epoch 4/20 - Loss: 0.7103
Epoch 5/20 - Loss: 0.4636
Epoch 6/20 - Loss: 0.2993
Epoch 7/20 - Loss: 0.1881
Epoch 8/20 - Loss: 0.1257
Epoch 9/20 - Loss: 0.0865
Epoch 10/20 - Loss: 0.0642
Epoch 11/20 - Loss: 0.0477
Epoch 12/20 - Loss: 0.0390
Epoch 13/20 - Loss: 0.0355
Epoch 14/20 - Loss: 0.0318
Epoch 15/20 - Loss: 0.0282
Epoch 16/20 - Loss: 0.0229
Epoch 17/20 - Loss: 0.0214
Epoch 18/20 - Loss: 0.0166
Epoch 19/20 - Loss: 0.0152
Epoch 20/20 - Loss: 0.0154
F1: 0.3883

--- BiGRU-CRF ---
Epoch 1/20 - Loss: 3.2323
Epoch 2/20 - Loss: 1.6062
Epoch 3/20 - Loss: 1.0653
Epoch 4/20 - Loss: 0.7045
Epoch 5/20 - Loss: 0.4579
Epoch 6/20 - Loss: 0.2934
Epoch 7/20 - Loss: 0.1891
Epoch 8/20 - Loss: 0.1275
Epoch 9/20 - Loss: 0.0868
Epoch 10/20 - Loss: 0.0640
Epoch 11/20 - Loss: 0.0520
Epoch 1

In [102]:
cross_dialect_results = pd.DataFrame(
    all_results
)

cross_dialect_results

,Held-out Dialect,Model,Precision,Recall,F1
0,Barishal,BiLSTM-CRF,0.282463,0.621180,0.388340
1,Barishal,BiGRU-CRF,0.217447,0.623815,0.322484
2,Barishal,Char-CNN-BiLSTM-CRF,0.574793,0.732877,0.644280
3,Chittagong,BiLSTM-CRF,0.169644,0.559151,0.260311
4,Chittagong,BiGRU-CRF,0.777778,0.497613,0.606923
5,Chittagong,Char-CNN-BiLSTM-CRF,0.800898,0.567639,0.664390
6,Mymensingh,BiLSTM-CRF,0.887364,0.824217,0.854626
7,Mymensingh,BiGRU-CRF,0.719434,0.837493,0.773988
8,Mymensingh,Char-CNN-BiLSTM-CRF,0.872940,0.872013,0.872476
9,Noakhali,BiLSTM-CRF,0.828086,0.699096,0.758144


In [103]:
summary = (
    cross_dialect_results
    .groupby("Model")["F1"]
    .agg(["mean", "std"])
    .reset_index()
)

summary

,Model,mean,std
0,BiGRU-CRF,0.641474,0.190359
1,BiLSTM-CRF,0.605809,0.263610
2,Char-CNN-BiLSTM-CRF,0.741440,0.093828


In [104]:
cross_dialect_results.to_csv(
    "/kaggle/working/cross_dialect_results.csv",
    index=False
)

summary.to_csv(
    "/kaggle/working/cross_dialect_summary.csv",
    index=False
)

print("Results saved.")

Results saved.
